# Analysis of generated samples

In [ ]:
import json
import re

import dotenv
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import openai
import pandas as pd
import pyrootutils
import seaborn as sns
import tiktoken

PROJECT_ROOT = path = pyrootutils.find_root(indicator=".project-root")
DATA_DIR = PROJECT_ROOT / "data"
BATCHES_DIR = PROJECT_ROOT / "batches"

dotenv.load_dotenv(PROJECT_ROOT / ".env")

In [ ]:
samples = []
samples_glob = DATA_DIR.glob("samples_*.jsonl")
for path in samples_glob:
    with open(path, "r") as f:
        samples.extend([json.loads(line) for line in f])

samples_df = pd.DataFrame(samples)

samples_df["input_length"] = samples_df["left_phonetic"].apply(lambda x: len(x.split()))

samples_df.info()

In [ ]:
ax = sns.histplot(samples_df["depth"], discrete=True)

In [ ]:
sns.histplot(samples_df["input_length"], discrete=True)

## Outputs

In [ ]:
openai_client = openai.OpenAI()

tokenizers = {"o4-mini": tiktoken.encoding_for_model("o4-mini")}

In [ ]:
outputs_glob = BATCHES_DIR.glob("*_output.jsonl")
inputs_glob = [
    path
    for path in BATCHES_DIR.glob("inputs_*.jsonl")
    if not path.name.endswith("_output.jsonl")
]

outputs = []
for path in outputs_glob:
    df = pd.read_json(path, lines=True)
    json_struct = json.loads(df.to_json(orient="records"))
    flat_df = pd.json_normalize(json_struct)

    batch_id = path.name.split("_output.jsonl")[0]

    # extract response
    flat_df["model_response"] = flat_df["response.body.choices"].apply(
        lambda x: x[0]["message"]["content"]
    )
    flat_df["prompt_tokens"] = flat_df["response.body.usage.prompt_tokens"]
    flat_df["completion_tokens"] = flat_df["response.body.usage.completion_tokens"]
    flat_df["total_tokens"] = flat_df["response.body.usage.total_tokens"]
    flat_df["model"] = flat_df["response.body.model"]
    flat_df["batch_id"] = batch_id

    outputs.append(flat_df)

outputs_df = pd.concat(outputs, ignore_index=True)

batch_ids = outputs_df["batch_id"].unique()
for bid in batch_ids:
    batch = openai_client.batches.retrieve(bid)
    input_file = openai_client.files.retrieve(batch.input_file_id)
    outputs_df.loc[outputs_df["batch_id"] == bid, "input_file"] = input_file.filename

inputs_dfs = []

for f in inputs_glob:
    df = pd.read_json(f, lines=True)
    json_struct = json.loads(df.to_json(orient="records"))
    flat_df = pd.json_normalize(json_struct)
    flat_df["input_file"] = f.name
    flat_df["grammar_name"] = flat_df.get("body.metadata.grammar_name")
    flat_df["input_sentence"] = flat_df.get("body.metadata.input_sentence")
    flat_df["output_sentence"] = flat_df.get("body.metadata.output_sentence")
    flat_df["depth"] = pd.to_numeric(
        flat_df.get("body.metadata.depth"), errors="coerce"
    )
    flat_df["n_rules"] = pd.to_numeric(
        flat_df.get("body.metadata.n_rules"), errors="coerce"
    )
    flat_df["n_words"] = pd.to_numeric(
        flat_df.get("body.metadata.n_words"), errors="coerce"
    )
    inputs_dfs.append(flat_df)

inputs_df = pd.concat(inputs_dfs, ignore_index=True)

merged_df = pd.merge(
    outputs_df,
    inputs_df,
    on=["input_file", "custom_id"],
    how="inner",  # or "outer" if you want rows even when one side is missing
    suffixes=("", "_other"),
)

In [ ]:
grammar_blobs = DATA_DIR.glob("grammar_*.json")
grammars = []

for path in grammar_blobs:
    with open(path, "r") as f:
        grammar = json.load(f)
        grammar_name = path.name.split("grammar_")[1].split(".json")[0]
        grammar["grammar_name"] = grammar_name
        grammar = pd.json_normalize(grammar)

        grammar["same_headedness"] = grammar["a.head_initial"].eq(
            grammar["b.head_initial"]
        )
        grammar["same_spec"] = grammar["a.spec_initial"].eq(grammar["b.spec_initial"])
        grammar["same_pro_drop"] = grammar["a.pro_drop"].eq(grammar["b.pro_drop"])
        grammar["same_prop_det"] = grammar["a.proper_with_det"].eq(
            grammar["b.proper_with_det"]
        )
        grammars.append(grammar)

grammars_df = pd.concat(grammars, ignore_index=True)

grammars_df["input_syllable_structure"] = grammars_df["a.syllable_structure"]
grammars_df["output_syllable_structure"] = grammars_df["b.syllable_structure"]

In [ ]:
grammars_df.columns

In [ ]:
# merge the lists of a.verbs, a.nouns, etc. into a list of all words in `a`, and
# similarly for `b`
grammars_df["a_words"] = grammars_df[
    [
        "a.verbs",
        "a.nouns",
        "a.propns",
        "a.prons",
        "a.adjs",
        "a.det_def",
        "a.det_indef",
        "a.comps",
    ]
].apply(lambda row: sum(row, []), axis=1)
grammars_df["b_words"] = grammars_df[
    [
        "b.verbs",
        "b.nouns",
        "b.propns",
        "b.prons",
        "b.adjs",
        "b.det_def",
        "b.det_indef",
        "b.comps",
    ]
].apply(lambda row: sum(row, []), axis=1)

grammars_df["b_words"]

In [ ]:
merged_df = pd.merge(
    merged_df,
    grammars_df[
        [
            "same_headedness",
            "same_spec",
            "same_pro_drop",
            "same_prop_det",
            "grammar_name",
            "input_syllable_structure",
            "output_syllable_structure",
            "a_words",
            "b_words",
        ]
    ],
    on="grammar_name",
)

merged_df["input_tokens"] = merged_df["input_sentence"].apply(
    lambda x: len(tokenizers["o4-mini"].encode(x)),
)
merged_df["output_tokens"] = merged_df["output_sentence"].apply(
    lambda x: len(tokenizers["o4-mini"].encode(x)),
)

In [ ]:
merged_df["input_syllable_structure"]

### Extract Answer

In [ ]:
answer_re = re.compile(r"Final answer: (.*?)(?:\n|$)", re.DOTALL)


def extract_answer(model_response):
    matches = answer_re.findall(model_response)
    if matches:
        last_match: str = matches[-1]
        last_match = re.sub(r"[^a-zA-Z\s]", "", last_match)
        last_match = last_match.strip()
        return last_match
    else:
        return None


# metrics
def exact_match(row) -> bool:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return False
    return row["model_answer"] == row["output_sentence"]


def bow_match(row) -> bool:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return False
    return sorted(row["model_answer"].split()) == sorted(row["output_sentence"].split())


def edit_distance(row) -> float:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return 1.0
    from strsimpy.jaro_winkler import JaroWinkler

    jw = JaroWinkler()
    return 1 - jw.distance(row["model_answer"], row["output_sentence"])


def num_oov_words(row) -> int:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return np.nan
    model_words = set(row["model_answer"].split())
    output_words = set(row["b_words"])
    oov_words = model_words - output_words
    print(len(oov_words))
    return len(oov_words)

In [ ]:
merged_df = merged_df.drop_duplicates(subset=["custom_id", "batch_id"])
merged_df["model_answer"] = merged_df["model_response"].apply(extract_answer)

# Metrics
merged_df["exact_match"] = merged_df.apply(exact_match, axis=1)
merged_df["bow_match"] = merged_df.apply(bow_match, axis=1)
merged_df["edit_distance"] = merged_df.apply(edit_distance, axis=1)
merged_df["num_oov_words"] = merged_df.apply(num_oov_words, axis=1)

merged_df["lhs_length"] = merged_df["input_sentence"].apply(lambda x: len(x.split()))
merged_df["depth"] = pd.to_numeric(merged_df.get("depth"), errors="coerce")
merged_df["n_rules"] = pd.to_numeric(merged_df.get("n_rules"), errors="coerce")
merged_df["n_words"] = pd.to_numeric(merged_df.get("n_words"), errors="coerce")
merged_df["size"] = merged_df["n_rules"] + merged_df["n_words"]

merged_df["ttc"] = merged_df["response.body.usage.completion_tokens"]
merged_df["ttc_binned"] = merged_df["ttc"].apply(
    # log2
    lambda x: round(np.log2(x)) if x > 0 else 0
)

merged_df["same_syllable_structure"] = (
    merged_df["input_syllable_structure"] == merged_df["output_syllable_structure"]
)

metrics_df = merged_df.melt(
    id_vars=[
        "model",
        "lhs_length",
        "custom_id",
        "model_answer",
        "output_sentence",
        "depth",
        "size",
        "same_headedness",
        "same_spec",
        "same_pro_drop",
        "same_prop_det",
        "ttc",
        "ttc_binned",
        "input_syllable_structure",
        "output_syllable_structure",
        "same_syllable_structure",
        "input_tokens",
        "output_tokens",
    ],
    value_vars=["exact_match", "bow_match", "edit_distance", "num_oov_words"],
    var_name="match_type",
    value_name="match_value",
)

# rename `exact_match` and `bow_match` in match_type column
metrics_df["match_type"] = metrics_df["match_type"].replace(
    {
        "exact_match": "Exact Match",
        "bow_match": "Bag of Words",
        "edit_distance": "Edit Distance",
        "num_oov_words": "Num OOV Words",
    }
)

metrics_df["input_tokens_rounded"] = metrics_df["input_tokens"].apply(
    lambda x: round(x / 5) * 5
)
metrics_df["output_tokens_rounded"] = metrics_df["output_tokens"].apply(
    lambda x: round(x / 5) * 5
)
metrics_df["token_delta"] = abs(
    metrics_df["output_tokens"] - metrics_df["input_tokens"]
)
metrics_df["token_delta_norm"] = abs(
    metrics_df["output_tokens"] - metrics_df["input_tokens"]
) / metrics_df["input_tokens"].replace(0, np.nan)
metrics_df["token_delta_norm_rounded"] = metrics_df["token_delta_norm"].apply(
    lambda x: round(x, 1)
)

# convert the `model` column to an ordered categorical type
metrics_df["model_name"] = metrics_df["model"].apply(lambda x: x.split("-2")[0])
metrics_df["model_name"] = pd.Categorical(
    metrics_df["model_name"],
    categories=["gpt-4.1-nano", "gpt-4.1-mini", "o4-mini"],
    ordered=True,
)

In [ ]:
merged_df["num_oov_words"]

### Plots

In [ ]:
ax = sns.lineplot(
    data=metrics_df[
        metrics_df["match_type"].isin(["Exact Match", "Bag of Words", "Edit Distance"])
    ],
    x="depth",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("Depth (# of embedded clauses)")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)

# format y axis as percentage for exact match and bow match

ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
ax = sns.lineplot(
    data=metrics_df[metrics_df["match_type"].isin(["Num OOV Words"])],
    x="lhs_length",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("Input length (# of words)")
ax.set_ylabel("Mean # of OOV Words")
# ax.set_ylim(0, 1)

# format y axis as percentage for exact match and bow match
# import matplotlib.ticker as mtick

# ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
ax = sns.lineplot(
    data=metrics_df,
    x="lhs_length",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("Sentence Length (# of words)")
ax.set_ylabel("Score")
# ax.set_ylim(0, 1)

# format y axis as percentage for exact match and bow match


ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
ax = sns.lineplot(
    data=metrics_df,
    x="input_tokens_rounded",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("Input Sentence Length (tokens)")
ax.set_ylabel("Score")
# ax.set_ylim(0, 1)

# format y axis as percentage for exact match and bow match


ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
ax = sns.lineplot(
    data=metrics_df,
    x="token_delta_norm_rounded",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("Token Delta")
ax.set_ylabel("Score")
# ax.set_ylim(0, 1)

# format y axis as percentage for exact match and bow match


ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
ax = sns.lineplot(
    data=metrics_df,
    x="output_tokens_rounded",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("Output Sentence Length (tokens)")
ax.set_ylabel("Score")
# ax.set_ylim(0, 1)

# format y axis as percentage for exact match and bow match


ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
ax = sns.lineplot(
    data=metrics_df,
    x="size",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("Grammar Size (# of productions)")
ax.set_ylabel("Score")
ax.set_ylim(0, None)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(8, 3))
spec = gridspec.GridSpec(ncols=3, nrows=1, figure=fig)

axes = [
    ax1 := fig.add_subplot(spec[0, 0]),
    ax2 := fig.add_subplot(spec[0, 1], sharey=ax1),
    ax3 := fig.add_subplot(spec[0, 2], sharey=ax1),
]

for i, col in enumerate(["same_headedness", "same_spec", "same_pro_drop"]):
    # pull out the axis that's already been created
    ax = axes[i]

    sns.barplot(
        data=merged_df,
        x=col,
        hue=col,
        y="exact_match",
        errorbar="se",
        ax=ax,
    )
    ax.set_title(col.replace("same_", "").replace("_", " ").title())

    counts = merged_df.groupby(col)["grammar_name"].nunique()
    for j, val in enumerate([False, True]):
        count = counts.get(val, 0)
        ax.text(
            j,
            0.05,
            f"N={count}",
            ha="center",
            va="bottom",
            fontsize=9,
            color="black",
            fontweight="bold",
        )

    if i > 0:
        ax.get_yaxis().set_visible(False)

    ax.set_ylabel("Mean Exact Match")
    ax.set_xlabel("Same" if col.startswith("same_") else col)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(8, 3))
spec = gridspec.GridSpec(ncols=2, nrows=1, figure=fig)

axes = [
    ax1 := fig.add_subplot(spec[0, 0]),
    ax2 := fig.add_subplot(spec[0, 1], sharey=ax1),
]

for i, col in enumerate(["input_syllable_structure", "output_syllable_structure"]):
    # pull out the axis that's already been created
    ax = axes[i]

    sns.barplot(
        data=merged_df,
        x=col,
        hue=col,
        y="exact_match",
        errorbar="se",
        ax=ax,
    )

    counts = merged_df.groupby(col)["grammar_name"].nunique()
    for j, val in enumerate(counts.index):
        count = counts.get(val, 0)
        ax.text(
            j,
            0.05,
            f"N={count}",
            ha="center",
            va="bottom",
            fontsize=9,
            color="black",
            fontweight="bold",
        )

    if i > 0:
        ax.get_yaxis().set_visible(False)

In [ ]:
ax = sns.barplot(
    data=merged_df,
    x="same_syllable_structure",
    hue="same_syllable_structure",
    y="exact_match",
    errorbar="se",
)

counts = merged_df.groupby("same_syllable_structure")["grammar_name"].nunique()
for j, val in enumerate([False, True]):
    count = counts.get(val, 0)
    ax.text(
        j,
        0.05,
        f"N={count}",
        ha="center",
        va="bottom",
        fontsize=9,
        color="black",
        fontweight="bold",
    )

### TTC

In [ ]:
ax = sns.lineplot(
    data=metrics_df,
    x="ttc_binned",
    y="match_value",
    hue="match_type",
    marker="o",
)

ax.set_xlabel("log2(Completion Tokens)  [binned]")
ax.set_ylabel("Score")
ax.set_ylim(0, None)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(5, 3))
spec = gridspec.GridSpec(ncols=2, nrows=1, figure=fig)

axes = [
    ax1 := fig.add_subplot(spec[0, 0]),
    ax2 := fig.add_subplot(spec[0, 1], sharey=ax1),
]

for i, col in enumerate(["depth", "size"]):
    ax = axes[i]
    sns.lineplot(data=metrics_df, x=col, y="ttc_binned", marker="o", ax=ax)
    ax.set_xlabel(col.title())
    ax.set_ylabel("log2(Completion Tokens) [binned]")
    if col == "size":
        ax.get_yaxis().set_visible(False)

### Error Analysis

In [ ]:
test = metrics_df[
    (metrics_df["match_type"] == "Exact Match") & (~metrics_df["match_value"])
].iloc[540][["model_answer", "output_sentence"]]

print(f'Model Answer:\n\t"{test["model_answer"]}"')
print(f'Output Sentence:\n\t"{test["output_sentence"]}"')

In [ ]:
test["model_answer"]

In [ ]:
test["output_sentence"]